# 0. Antes de empezar

Definimos aquí algunas funciones y clases que utilizaremos en esta Notebook

In [ ]:
pip install rasterio

In [ ]:
#Traemos funciones de la clase anterior

import numpy as np

def escalar_imagen(imagen, p):
  # Calcular los percentiles p y 100-p
  valor_min = np.percentile(imagen, p)
  valor_max = np.percentile(imagen, 100-p)

  # Truncar la imagen al rango [p1, p99]
  ecualizada = np.clip(imagen, valor_min, valor_max)

  # Normalizar la imagen al rango 0-255.
  ecualizada = (ecualizada - valor_min) / (valor_max - valor_min) * 255

  # Convertir la imagen ecualizada a entero de 8 bits
  ecualizada = np.uint8(ecualizada)

  # Retornar la imagen ecualizada
  return ecualizada

def escalar_imagen_multibanda(imagen, p):
  # Crear un array vacio con la misma estructura que imagen_spot_apilada
  imagen_escalada = np.empty_like(imagen)

  # Definir una variable con el número de canales
  canales = imagen_escalada.shape[-1]

  # Escalar una a una las bandas (usando p)
  for i in range(canales):
    canal = imagen[:, :, i]
    imagen_escalada[:, :, i] = escalar_imagen(canal, p)

  # Convertir a np.uint8 el array escalado (no hace falta seguir usando float)
  imagen_escalada = np.uint8(imagen_escalada)

  return imagen_escalada

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab.output import eval_js
from base64 import b64encode
from PIL import Image
import io
from shapely.geometry import box
import rasterio
import rasterio.transform

class Extractor():
    def __init__(self, img_np, img_rgb, metadata) -> None:
        self.extracted_dataframe = pd.DataFrame()
        self.extracted_polygons = gpd.GeoDataFrame()  # GeoDataFrame para polígonos georreferenciados
        self.img_np = img_np
        self.img_rgb = img_rgb
        self.metadata = metadata  # Diccionario de metadatos de rasterio
        self.callback = self.manejar_df_extraido

    def manejar_df_extraido(self, df, polygons):
        self.extracted_dataframe = df.copy()
        self.extracted_polygons = polygons  # Guardar el GeoDataFrame de polígonos extraídos

    def interfaz_seleccion_pixeles(self):
        scale_factor = 2  # Factor de escalado (aumenta el tamaño de la imagen)
        img_pil = Image.fromarray(self.img_rgb)
        buffered = io.BytesIO()
        img_pil.save(buffered, format="PNG")
        img_b64 = b64encode(buffered.getvalue()).decode()

        new_width = self.img_rgb.shape[1] * scale_factor
        new_height = self.img_rgb.shape[0] * scale_factor

        js_code = f"""
        <style>
          canvas {{ border: 3px solid black; display: block; margin: auto; }}
        </style>
        <canvas id="canvas" width="{new_width}" height="{new_height}"></canvas>
        <script>
          var scaleFactor = {scale_factor};
          var canvas = document.getElementById('canvas');
          var ctx = canvas.getContext('2d');
          var img = new Image();
          img.src = "data:image/png;base64,{img_b64}";
          img.onload = function() {{
              ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
          }};

          var rects = [];
          var startX, startY, drawing = false;

          canvas.addEventListener('mousedown', function(e) {{
              startX = e.offsetX / scaleFactor;
              startY = e.offsetY / scaleFactor;
              drawing = true;
          }});

          canvas.addEventListener('mouseup', function(e) {{
              if (!drawing) return;
              drawing = false;
              var endX = e.offsetX / scaleFactor;
              var endY = e.offsetY / scaleFactor;
              rects.push([startX, startY, endX, endY]);

              ctx.strokeStyle = 'red';
              ctx.lineWidth = 2;
              ctx.strokeRect(startX * scaleFactor, startY * scaleFactor,
                             (endX - startX) * scaleFactor,
                             (endY - startY) * scaleFactor);
          }});

          function getRects() {{ return rects; }}
          function clearRects() {{
              rects = [];
              ctx.clearRect(0, 0, canvas.width, canvas.height);
              ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
          }}
        </script>
        """
        display(HTML(js_code))

        def get_rectangles():
            return eval_js("getRects()")

        def extract_pixels():
            rects = get_rectangles()
            if not rects:
                print("No rectangles drawn.")
                return pd.DataFrame(), gpd.GeoDataFrame()

            dataframes = []
            extracted_polygons = []
            transform = self.metadata.get("transform")
            crs = self.metadata.get("crs")

            for i, (x1, y1, x2, y2) in enumerate(rects):
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # Convertir a coordenadas georreferenciadas
                lon1, lat1 = rasterio.transform.xy(transform, y1, x1, offset='center')
                lon2, lat2 = rasterio.transform.xy(transform, y2, x2, offset='center')
                polygon = box(lon1, lat1, lon2, lat2)
                extracted_polygons.append(polygon)

                # Extraer valores de todas las bandas
                selected_pixels = self.img_np[:, y1:y2, x1:x2]
                pixel_values = selected_pixels.reshape(selected_pixels.shape[0], -1).T
                df = pd.DataFrame(pixel_values, columns=[f"Band_{b}" for b in range(self.img_np.shape[0])])
                df["X"] = np.tile(np.arange(x1, x2), y2 - y1)
                df["Y"] = np.repeat(np.arange(y1, y2), x2 - x1)
                df["Rect_ID"] = i + 1
                dataframes.append(df)

            # Crear un GeoDataFrame con los polígonos y asignar el CRS
            gdf = gpd.GeoDataFrame(geometry=extracted_polygons, crs=crs)

            return pd.concat(dataframes, ignore_index=True) if dataframes else pd.DataFrame(), gdf

        extract_button = widgets.Button(description="Extraer Valores")
        clear_button = widgets.Button(description="Borrar Todo")
        output = widgets.Output()

        def on_extract_clicked(b):
            with output:
                output.clear_output()
                extracted_df, extracted_polygons = extract_pixels()
                if self.callback:
                    self.manejar_df_extraido(extracted_df, extracted_polygons)

        def on_clear_clicked(b):
            eval_js("clearRects()")

        extract_button.on_click(on_extract_clicked)
        clear_button.on_click(on_clear_clicked)

        display(extract_button, clear_button, output)
        return None


# Notebook 02a: Imágenes satelitales

### Datos con los que trabajaremos en esta notebook:

- Una imagen Sentinel-2 de la misma zona de estudio (**[S2_20180619_2-3-4-8-11-12.tif](https://drive.google.com/file/d/1dehVX2ClJBhEhGxrrVhvS5LH3SNK4RL0/view?usp=sharing)**).
- Una imagen del isntrumento Moderate-Resolution Imaging Spectroradiometer (MODIS) preparada para ver ejemplos de visualización **[MOD09A1.A2000049.h11v08.006.2015136143404_b01_02_03.tif](https://drive.google.com/file/d/1NcAVXobn2u0CMB0mPgepdPrfE6ovGSLz/view?usp=drive_link)**

Pueden descargarlas en sus equipos para visualizarlas localmente, o bien usar `gdown` para bajarlas directamente en la sesión de Colab.

## 1. Lectura de una imagen satelital y sus metadatos

Para leer imágenes georreferenciadas podemos usar principalmente dos librerías: `GDAL` o `Rasterio`. `GDAL` (Geospatial Data Abstraction Library) es una biblioteca de código abierto ampliamente utilizada para el procesamiento y manipulación de datos geoespaciales. En Python, GDAL se accede a través del paquete osgeo.gdal, que permite leer, escribir y convertir formatos raster y vectoriales. Se parece más a una librería clásica de `C` con lo cual implica trabajar o progamar más en bajo nivel. `Rasterio` viene a ser la adaptación de GDAL para las convenciones de Python: Usa esta librería como base, pero permite al usuario trabajar a más alto nivel. Ofrece algunas ventajas en cuanto a funcionalidades que ahorran mucho trabajo al usuario, y usaremos algunas de estas en esta clase.


### 1.1. Lectura de archivos raster con Rasterio

Para comenzar, levantaremos nuestra imagen Sentinel-2 (archivo **S2_20180619_2-3-4-8-11-12.tif**) con Rasterio y leeremos algunas de sus propiedades para entender cómo las interpreta esta biblioteca

La imagen Sentinel 2 que hemos preparado para esta notebook contiene 6 bandas:

- Banda 1: Azul
- Banda 2: Verde
- Banda 3: Rojo
- Banda 4: Infrarrojo Cercano (NIR)
- Banda 5: Infrarrojo de Onda Corta 1 (SWIR 1)
- Banda 6: Infrarrojo de Onda Corta 2 (SWIR 2).

En la siguiente tabla se resaltan en negrita con su orden original dentro de una Sentinel-2. En la Notebook 02c se muestra cómo extraer un archivo Sentinel-2 de cero.

| Banda de Sentinel-2  | Rango de Longitudes de Onda (nm) | Centro de Banda (nm) | Banda en el GeoTiff |
|--------|----------------------------------|----------------------|---------------------|
| Banda 1 | 443 - 453                       | 448                  | N/A                 |
| **Banda 2** | **490 - 510**                       | **500**                  | **1**                  |
| **Banda 3** | **560 - 580**                       | **570**                  | **2**                  |
| **Banda 4** | **665 - 680**                       | **672.5**                | **3**                  |
| Banda 5 | 705 - 745                       | 725                  | N/A                 |
| Banda 6 | 740 - 780                       | 760                  | N/A                 |
| Banda 7 | 773 - 793                       | 783                  | N/A                 |
| **Banda 8** | **785 - 900**                       | **842.5**                | **4**                  |
| Banda 8A| 855 - 875                       | 865                  | N/A                 |
| Banda 9 | 935 - 955                       | 945                  | N/A                 |
| Banda 10| 1360 - 1390                      | 1375                 | N/A                 |
| **Banda 11**| **1610 - 1650**                      | **1630**                 | **5**                  |
| **Banda 12**| **2190 - 2250**                      | **2170**                 | **6**                  |


Más especificaciones sobre las bandas de Sentinel 2 se pueden encontrar en [este enlace](https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-2-msi/resolutions/spatial)

```python
import gdown
import rasterio

# ID del archivo de Google Drive
file_id = "1dehVX2ClJBhEhGxrrVhvS5LH3SNK4RL0"
url = f"https://drive.google.com/uc?id={file_id}"

# Descargar la imagen
output = "S2_20180619_2-3-4-8-11-12.tif"
gdown.download(url, output, quiet=False)

# Cargar y mostrar la imagen
s2 = rasterio.open(output)
print(f'El tipo de objeto al abrir con Rasterio es: {type(s2)}')
```

Como se puede ver, tanto Rasterio interpreta a las imágenes como clases, sobre las que podremos aplicar ciertos métodos y que tendrán ciertos atributos. En esos métodos y atributos radican las principales diferencias entre ambas librerías. Veamos como se leen algunos de los principales atributos:

```python
#Proyección
print(s2.crs)

#Geotransform:
print(s2.transform)

# Dimensiones y cantidad de bandas
print(s2.width)
print(s2.height)
print(f'Número de bandas: {s2.count}')

#Extensión:
print(s2.bounds)

#Metadatos:
print(s2.meta)
```

En este último parámetro (metadatos) se resume toda la información del Raster, con estructura de diccionario. Cuando escribamos imágenes modificadas vamos a editar principalmente este atributo.

**¿Qué es un diccionario?**  Los diccionarios son estructuras de datos que, al igual que las listas y las tuplas, permiten almacenar diferentes tipos de datos, sin embargo, a diferencia de estas los elementos no son indexados, sino que se almacenan de la forma llave y valor, donde la llave corresponde a un identificador único que permite acceder al elemento. Un diccionario se define entre llaves {}, cada uno de sus elementos se encuentra separado por comas (,) y cada par {llave: valor} se separa por dos puntos (:)

![diccionario_1.png](https://drive.google.com/uc?id=1yaOCC_NQtZaREBGpL6QSgH44gYtIylGs)

Aquí va una vista más ordenada del diccionario de metadatos de nuestra imagen:

```python
{'driver': 'GTiff',
 'dtype': 'uint16',
 'nodata': 0.0,
 'width': 1080,
 'height': 593,
 'count': 6,
 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 20S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-63],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32720"]]'),
 'transform': Affine(10.0, 0.0, 388870.0,
       0.0, -10.0, 6528050.0)}
```

La información que tenemos en este diccionario es la siguiente:

- driver: El tipo o formato de archivo
- dtype: El tipo de dato de los píxeles
- nodata: El valor de relleno o valor considerado como _no dato_ o no válido.
- width: Ancho o cantidad de columnas
- height: Alto o cantidad de filas
- count: Cantidad de bandas o canales
- crs: Descripción del sistema de coordenadas de la imagen
- transform: Es la Geotrasnformación, leía como matríz _afín_.

Para leer las bandas como matrices o arreglos de `NumPy`, se usa el método `read`:

```python
s2_bandas = s2.read()

print(f"Dimensiones de la imagen leída como array: {s2_bandas.shape}")
```

### 1.2. Visualización en RGB - Uso de funciones anteriores

Las funciones que traemos de la clase anterior asumen que la matriz de entrada tiene una estructura `(filas,columnas,bandas)`. Sin embargo, como se puede ver de la celda anterior, `Rasterio` lee estas matrices como `(bandas,filas,columnas)`. Para utilizar estas funciones de escalado debemos trasponer los ejes.

```python
from matplotlib import pyplot as plt

# Aplicar la función sobre la matriz traspuesta
s2_bandas_escalado = escalar_imagen_multibanda(s2_bandas.transpose(1,2,0), 1)

#plotear el RGB
plt.figure()
plt.imshow(s2_bandas_escalado[:,:,[2,1,0]])
plt.show()
```

`imshow()` pone por defecto los números de fila y columna en los ejes del gráfico. Sin embargo, si queremos tener una referencia cartográfica, podemos usar las coordenadas que nos da `bounds` y pasárselas al ploteo usando el parámetro `extent`.

```python
left, bottom, right, top = s2.bounds  # Extraer los límites

# Mostrar la imagen con los límites geoespaciales
plt.figure()
plt.imshow(s2_bandas_escalado[:, :, [2, 1, 0]], extent=[left, right, bottom, top], origin="upper")
plt.ticklabel_format(style="plain", axis="y")
plt.show()
```

La línea `plt.ticklabel_format(style="plain", axis="y")` se utiliza para que la coordenada se imprima entera, de lo contrario queda escalada por `1e6` (1 millón).

Si queremos tener información un poco más detallada sobre el sistema de coordenadas `EPSG:32720` que tiene esta imagen, podemos hacer:

```python
print(f'Descripción de {s2.crs}')
print(s2.crs.to_wkt())
```

### 1.3. Visualización - Tratamiento de los valores de relleno o _no válidos_

Es imágenes satelitales es muy habitual que haya zonas sin datos válidos, que se rellenan con algún valor específico, como en el siguiente ejemplo:


![modis.png](https://drive.google.com/uc?id=1kJ1-OAe0w5PkeF9GKDaXw7fAslv0rKPH)

Esto afecta al escalado de las imágenes, dado que se basa en las estadísticas de las imágenes, y muchas veces este valor de relleno es un número alto o bajo, que influye en el cálculo de los percentiles, la media, el desvío estándar, etc. Vamos a abrir un archivo GeoTiff preparado para ilustrar este ejemplo, que corresponde originalmente a las bandas 01, 02 y 03 de una adquisición del instrumento [MODIS](https://catalogos.conae.gov.ar/TERRA/camara.htm):

```python
# ID del archivo de Google Drive
file_id = "1NcAVXobn2u0CMB0mPgepdPrfE6ovGSLz"
url = f"https://drive.google.com/uc?id={file_id}"

# Descargar la imagen
output = "MOD09A1.A2000049.h11v08.006.2015136143404_b01_02_03.tif"
gdown.download(url, output, quiet=False)
```

Si lo abrimos con Rasterio y miramos sus metadatos, vamos a advertir que el valor de `nodata` es `-28762`.

```python
# Cargar y mostrar la imagen
modis = rasterio.open(output)

print("Metadatos del archivo MODIS")
for key, value in modis.meta.items():
    print(f"{key}: {value}")
```

Cuando escalamos la imagen, si no tenemos en cuenta este dato, no vamos a obtener un buen resultado:

```python
# Leer la imagen como array y aplicar la función sobre la matriz traspuesta
modis_bandas = modis.read()
modis_bandas_escalado = escalar_imagen_multibanda(modis_bandas.transpose(1,2,0), 1)

# plotear el RGB
plt.figure()
plt.imshow(modis_bandas_escalado[:,:,[2,1,0]])
plt.show()
```

Entonces, necesitamos modificar estas funciones para que en el cálculo de los percentiles, este valor se vea reflejado. Aquí va un código con esta versión modificada:

```python
def escalar_imagen(imagen, p, nodata = None):
  # Calcular los percentiles p y 100-p
  valor_min = np.percentile(imagen[imagen != nodata], p)
  valor_max = np.percentile(imagen[imagen != nodata], 100-p)

  # Truncar la imagen al rango [p1, p99]
  ecualizada = np.clip(imagen, valor_min, valor_max)

  # Normalizar la imagen al rango 0-255.
  ecualizada = (ecualizada - valor_min) / (valor_max - valor_min) * 255

  # Convertir la imagen ecualizada a entero de 8 bits
  ecualizada = np.uint8(ecualizada)

  # Retornar la imagen ecualizada
  return ecualizada

def escalar_imagen_multibanda(imagen, p, nodata = None):
  # Crear un array vacio con la misma estructura que imagen_spot_apilada
  imagen_escalada = np.empty_like(imagen)

  # Definir una variable con el número de canales
  canales = imagen_escalada.shape[-1]

  # Escalar una a una las bandas (usando p)
  for i in range(canales):
    canal = imagen[:, :, i]
    imagen_escalada[:, :, i] = escalar_imagen(canal, p, nodata)

  # Convertir a np.uint8 el array escalado (no hace falta seguir usando float)
  imagen_escalada = np.uint8(imagen_escalada)

  return imagen_escalada
```

¿Qué cambió en estas funciones=

- Se agregó un parámetro `nodata`. Por **defecto**, su valor es `None`. Es decir que, si no se le especifica ningún valor, la función asume que no hay valor de `nodata`.
- El parámetro `nodata` se agrega tanto en `escalar_imagen` como en `escalar_imagen_multibanda`. Cuando llamamos a `escalar_imagen` dentro de la otra función, se debe pasar este valor como argumento. Por ello la línea modificada: `imagen_escalada[:, :, i] = escalar_imagen(canal, p, nodata)`.
- Al calcular los percentiles, se consideran solamente aquellos píxeles que sean distintos de `nodata`, al usar `imagen[imagen != nodata]`.

In [ ]:
# Valor nodata de la NODIS
modis.nodata

Ahora, vamos a probar de nuevo las funciones con estos cambios:

```python
# Aplicar la función sobre la matriz traspuesta
modis_bandas_escalado = escalar_imagen_multibanda(modis_bandas.transpose(1,2,0), 1, nodata = modis.nodata)

# Plotear el RGB
plt.figure()
plt.imshow(modis_bandas_escalado[:,:,[2,1,0]])
plt.show()
```

Ahora el resultado se ve mucho mejor. Un par de aclaraciones sobre lo que ha ocurrido en la celda anterior:

- En lugar de pasarle a la función el valor `-28672`, se le ha pasado directamente el valor de la variable `modis.nodata`. Esto es debido a que los valores en ``float32` pueden sufrir errores de precisión. Si `nodata` fue leído desde un archivo con rasterio, es posible que su valor real no sea exactamente `-28672.0`, sino algo muy cercano.
- Toda la zona cubierta por el valor de relleno quedó oscura porque la función lo llevó a 0, ya que `-28672.0`es menor que `valor_min`. Sin embargo, esto no siempre va a ser así. Si el valor de relleno por ejemplo fuera `99999`, en la imagen escalada el valor se vería blanco, ya que quedaría con valor `255`.

## 2. Combinaciones RGB e interpretación espectral

En esta sección de la notebook visualizaremos la imagen Sentinel-2 con distintas combinaciones RGB, y graficaremos el patrón espectral para distintos tipos de cobertura del suelo, ayudándonos de la _clase_ `Extractor()` que nos permite marcar de manera interactiva zonas de interés y extraer información.

### 2.1. Visualizar las coberturas del suelo con distintas combinaciones RGB

Tener presente la zona que se está estudiando. Corresponde a las afueras de la Ciudad de Córdoba (Argentina), donde podemos ver usos y ocupaciones del suelo muy diversas:

![spotUsos.png](https://drive.google.com/uc?id=1ui90bRtbRtr39pLtAlzAE847ytREls_L)

Comencemos por visualizar la imagen en **Color Natural**. Esto equivale a usar la combinación `[2,1,0]`. Podemos utilizar la variable que ya tenemos definida, `s2_bandas_escalado`. Como ayuda memoria, estas son las bandas contenidas en esta imagen Sentinel-2:

- Banda 1: Azul
- Banda 2: Verde
- Banda 3: Rojo
- Banda 4: Infrarrojo Cercano (NIR)
- Banda 5: Infrarrojo de Onda Corta 1 (SWIR 1)
- Banda 6: Infrarrojo de Onda Corta 2 (SWIR 2).

Para color natural, así armamos esta combinación:

**🔴 Canal Rojo**: Banda 3 (en Numpy es la 2)  
**🟢 Canal Verde**: Banda 2 (en Numpy es la 1)  
**🔵 Canal Azul**: Banda 1 (en Numpy es la 0)  


```python
#plotear el RGB
plt.figure(figsize = (15,9))
plt.imshow(s2_bandas_escalado[:,:,[2,1,0]])
plt.axis("off")
plt.show()
```




A continuación probamos con la combinación **Falso Color Infrarrojo (Vegetación)**

**🔴 Canal Rojo**: Banda 4 (en Numpy es la 3)  
**🟢 Canal Verde**: Banda 3 (en Numpy es la 2)  
**🔵 Canal Azul**: Banda 2 (en Numpy es la 1)  


```python
# plotear el RGB
plt.figure(figsize = (15,9))
plt.imshow(s2_bandas_escalado[:,:,[3,2,1]])
plt.axis("off")
plt.show()
```

Por último, probemos con la combinación **Falso Color SWIR** mostrada en el teórico:

**🔴 Canal Rojo**: Banda 5 (en Numpy es la 4)  
**🟢 Canal Verde**: Banda 4 (en Numpy es la 3)  
**🔵 Canal Azul**: Banda 3 (en Numpy es la 2)  

```python
# plotear el RGB
plt.figure(figsize = (15,9))
plt.imshow(s2_bandas_escalado[:,:,[4,3,2]])
plt.axis("off")
plt.show()
```

Luego de mirar las tres imágenes, vamos a discutir algunas de estas preguntas:

- El agua va viendose de distintas tonalidades a medida que cambiamos la combinación de colores, ¿a qué puede deberse?
- ¿Por qué vemos la vegetación en rojo en una combinación, y en otras la vemos en tonos de verde?
- ¿Qué sucede con el suelo desnudo (aparentemente sin vegetación ni construcciones), qué tonalidad va tomando en cada combinación?
- ¿Porqué ciertos techos y/o construcciones artificiales se ven siempre más bien brillantes?

### 2.2 Extraer valores de píxeles y graficar los patrones espectrales de un tipo de supercicie

En esta sección usaremos la clase `Extractor`. Esta clase que hemos definido toma de entrada un _array_ conteniendo la imagen, y otro escalado con nuestras funciones para poner como RGB de fondo. Allí podemos jugar con distintas combinaciones de bandas para entender cómo cambia el color de las disintas superficies según las bandas del espectro que elegimos. Por último, toma como entrada el diccionario con los _metadatos_ de la imagen leía por Rasterio. Esto nos servirá para la siguiete Notebook, donde trabajaremos con polígonos y recortes.

Para usar esta clase debemos llamarla como si fuera una función, y pasarle los siguientes parámetros:

- El `array`representando la imagen (sin escalar), en este caso `s2_bandas`
- El mismo `array`escalado, en este caso usaremos `s2_bandas_escalado`, pero eligiendo la combinación RGB que queremos poner de fondo. Por ejemplo, le podemos pasar `s2_bandas_escalado[:,:,[5,4,3]]`.
- El diccionario conteniendo los metadatos, que se usarán para crear los polígonos. de los rectángulos.

El código sería el siguiente:

```python
seleccion_pixeles = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
```

Esto solamente _inicializa_ este objeto representado por la clase. Para que se nos cargue la interfaz que nos permite marcar los recántulos, debemos hacer:

```python
seleccion_pixeles.interfaz_seleccion_pixeles()
```

Donde `interfaz_seleccion_pixeles()`es uno de los métodos de la clase `Extractor()` que nos permite dibujar de manera interactiva:

Vamos a probarlo:


In [ ]:
seleccion_pixeles = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles.interfaz_seleccion_pixeles()

Una vez que ejecutes la celda anterior, vas a poder marcar rectánculos. Tratá de definir zonas homogéneas. Por ejemplo, el cuerpo de agua que se marca en la siguiente captura:

![spotUsos.png](https://drive.google.com/uc?id=1bZtJUz2puY9vZM4kNV4DsiuamVK2jevh)

Y luego hacé click en **extraer valores**. También poodés marcar más de un rectángulo, y se puede borar todo si hubo algún error.

El objetivo es extraer valores de píxeles del mismo tipo de cobertura (por ejemplo, vegetación, agua, zonas construidas, suelo desnudo), para luego ver sus firmas.

Una vez que hayas extraído los valores, podés usar el método `.extracted_dataframe()` y esto mostrará una tabla más o menos similar a la siguiente (seguramente no tendrás los mismos valores):

|index|Band\_0|Band\_1|Band\_2|Band\_3|Band\_4|Band\_5|X|Y|Rect\_ID|
|---|---|---|---|---|---|---|---|---|---|
|0|1122|1217|713|548|301|215|677|332|2|
|1|1137|1226|716|543|301|215|678|332|2|
|2|1132|1211|725|565|317|216|679|332|2|
|3|1137|1236|731|566|317|216|680|332|2|
|4|1147|1230|726|573|314|223|681|332|2|
|566|980|837|566|490|285|245|654|333|5|
|...|...|...|...|...|...|...|...|...|...|
|568|970|850|564|509|255|231|649|334|5|
|569|996|839|556|512|255|231|650|334|5|
|570|995|845|556|505|260|233|651|334|5|
|571|993|848|559|498|260|233|652|334|5|
|572|979|842|567|497|264|229|653|334|5|
|573|974|849|565|506|264|229|654|334|5|

Lo que se está mostrando es una tabla en formato de [**_DataFrame_** de _Pandas_](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html), donde se tiene el valor de los píxeles contenidos en los recángulos en cada una de las bandas (0 a 5 en este caso), las coordenadas de cada píxel (fila y columna) y un identificador de rectángulo, en el caso que hayas dibujado más de uno.

```python
seleccion_pixeles.extracted_dataframe
```

In [ ]:
seleccion_pixeles.extracted_dataframe

Podemos aprovechar esta estructura para hacer rápidamente algunas cuentas. En primer lugar, vamos a calcular la media y los desvíos estándar para todas las bandas. Luego, usando `to_numpy()` convertiremos eas medias y desvíos en `arrays` de `NumPy`:

```python
# Calcular la media y el desvío de cada banda
# Calcular la media y el desvío de cada banda
df = seleccion_pixeles.extracted_dataframe.copy()

bandas_promedio = df[['Band_0', 'Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5']].mean().to_numpy()
bandas_std = df[['Band_0', 'Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5']].std().to_numpy()
```

Y a continuación podemos plotear el patrón espectral de estos píxeles, poniendo en el eje horizonal los centros de onda a los que corresponde cada valor. Estos son los centros de londigud de onda de cada banda de Sentinel-2, donde hemos resaltado en negrita las bandas con las cuales estamos trabajando en esta Notebook:

| Banda de Sentinel-2  | Rango de Longitudes de Onda (nm) | Centro de Banda (nm) | Banda en el GeoTiff |
|--------|----------------------------------|----------------------|---------------------|
| Banda 1 | 443 - 453                       | 448                  | N/A                 |
| **Banda 2** | **490 - 510**                       | **500**                  | **1**                  |
| **Banda 3** | **560 - 580**                       | **570**                  | **2**                  |
| **Banda 4** | **665 - 680**                       | **672.5**                | **3**                  |
| Banda 5 | 705 - 745                       | 725                  | N/A                 |
| Banda 6 | 740 - 780                       | 760                  | N/A                 |
| Banda 7 | 773 - 793                       | 783                  | N/A                 |
| **Banda 8** | **785 - 900**                       | **842.5**                | **4**                  |
| Banda 8A| 855 - 875                       | 865                  | N/A                 |
| Banda 9 | 935 - 955                       | 945                  | N/A                 |
| Banda 10| 1360 - 1390                      | 1375                 | N/A                 |
| **Banda 11**| **1610 - 1650**                      | **1630**                 | **5**                  |
| **Banda 12**| **2190 - 2250**                      | **2170**                 | **6**                  |

Y a continuación el código para el gráfico del patrón espectral:

```python
# Longitudes de onda (en nanómetros) de las bandas Sentinel-2 utilizadas
wavelengths_nm = np.array([490, 560, 665, 842, 1610, 2190])  # Bandas 2, 3, 4, 8, 11, 12

# Graficar las medias de las bandas
plt.figure(figsize=(8, 5))
plt.plot(wavelengths_nm, bandas_promedio, 's-', label="Reflectancia Promediada para el agua", color='blue')

# Agregar la banda de tolerancia (media +/- 1 desviación estándar)
plt.fill_between(wavelengths_nm, bandas_promedio - bandas_std, bandas_promedio + bandas_std,
                 color='blue', alpha=0.3)

# Configuración del gráfico
plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Reflectancia (promediada)")
plt.title("Patrones espectrales Promediados de Sentinel-2")
plt.legend()
plt.grid(True)
plt.show()
```

Puede ser de utilidad mirar este gráfico del patrón espectral junto a distintas combinaciones RGB, y de esta manera entender por qué se dan ciertos colores en la superficie de la cual se extrajeron datos. Dejamos la siguien celda de ejemplo, donde se puede elegir la combinación de bandas, y luego graficar la imagen RGB con esa combinación y la gŕafica con el patrón espectral de los datos extraídos:

```python
# Definir bandas RGB para la imagen
bandas_rgb = [4, 3, 2]  # Índices correspondientes a  Índices correspondientes a SWIR1, NIR, Rojo

# Crear figura con dos subgráficos
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# --- Subplot 1: Imagen Sentinel-2 con bandas RGB ---
ax[0].imshow(s2_bandas_escalado[:, :, bandas_rgb])  # Mostrar imagen con bandas seleccionadas
ax[0].axis("off")  # Quitar ejes
ax[0].set_title("Imagen RGB de Sentinel-2")

# --- Subplot 2: Gráfico de firmas espectrales ---
ax[1].plot(wavelengths_nm, bandas_promedio, 'o-', label="Media espectral", color='black')

# Agregar banda de tolerancia (±1 desviación estándar)
ax[1].fill_between(wavelengths_nm, bandas_promedio - bandas_std, bandas_promedio + bandas_std,
                   color='gray', alpha=0.3, label="±1 Desviación estándar")

# Dibujar líneas verticales para las bandas seleccionadas (según bandas_rgb)
colors = ['red', 'green', 'blue']
for i, band in enumerate(bandas_rgb):
    ax[1].axvline(x=wavelengths_nm[band], color=colors[i], alpha=0.3, linewidth=8)

# Configuración del gráfico espectral
ax[1].set_xlabel("Longitud de onda (nm)")
ax[1].set_ylabel("Valor promedio de la banda")
ax[1].set_title("Patrón espectral con bandas seleccionadas")
ax[1].legend()
ax[1].grid(True)

# Mostrar el gráfico
plt.tight_layout()
plt.show()
```


### 2.3. Extraer valores de píxeles de varias coberturas y comparar sus patrones espectrales.

Para cerrar esta notebook, vamos a extraer valores de píxeles para vegetación, suelo desnudo y zona urbana o construída. Hay que repetir los pasos realizados con el agua, pero ir guadándolos a variables diferentes. Por ejemplo, podés ir corriendo en celdas separadas los siguientes bloques:

```python
# Vegetacion
seleccion_pixeles_vegetacion = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_vegetacion.interfaz_seleccion_pixeles()

# Suelo desnudo
seleccion_pixeles_suelo = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_suelo.interfaz_seleccion_pixeles()

# Suelo construido
seleccion_pixeles_construido = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_construido.interfaz_seleccion_pixeles()
```

Luego, obtener las medías y desvíos usando esta función para no repetir tanto los comandos:

```python
def obtener_firmas(seleccion_pixeles):
  df = seleccion_pixeles.extracted_dataframe.copy()
  bandas_promedio = df[['Band_0', 'Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5']].mean().to_numpy()
  bandas_std = df[['Band_0', 'Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5']].std().to_numpy()

  return bandas_promedio, bandas_std

bandas_promedio_vegetacion, bandas_std_vegetacion = obtener_firmas(seleccion_pixeles_vegetacion)
bandas_promedio_suelo, bandas_std_suelo = obtener_firmas(seleccion_pixeles_suelo)
bandas_promedio_construido, bandas_std_construido = obtener_firmas(seleccion_pixeles_construido)

```

In [ ]:
# Vegetacion
seleccion_pixeles_vegetacion = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_vegetacion.interfaz_seleccion_pixeles()

In [ ]:
# Suelo desnudo
seleccion_pixeles_suelo = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_suelo.interfaz_seleccion_pixeles()

In [ ]:
# Suelo construido
seleccion_pixeles_construido = Extractor(s2_bandas, s2_bandas_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_construido.interfaz_seleccion_pixeles()

Y finalmente, plotear todos los patrones espectrales juntos:

```python
# Definir bandas RGB para la imagen
bandas_rgb = [4, 3, 2]  # Índices correspondientes a SWIR1, NIR, Rojo

# Crear figura con dos subgráficos
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# --- Subplot 1: Imagen Sentinel-2 con bandas RGB ---
ax[0].imshow(s2_bandas_escalado[:, :, bandas_rgb])  # Mostrar imagen con bandas seleccionadas
ax[0].axis("off")  # Quitar ejes
ax[0].set_title("Imagen RGB de Sentinel-2")

# --- Subplot 2: Gráfico de firmas espectrales ---
ax[1].plot(wavelengths_nm, bandas_promedio, 'o-', label="Agua", color='blue')
ax[1].plot(wavelengths_nm, bandas_promedio_vegetacion, 'o-', label="Vegetacion", color='green')
ax[1].plot(wavelengths_nm, bandas_promedio_suelo, 'o-', label="Suelo", color='yellow')
ax[1].plot(wavelengths_nm, bandas_promedio_construido, 'o-', label="Construido", color='violet')

# Agregar banda de tolerancia (±1 desviación estándar)
ax[1].fill_between(wavelengths_nm, bandas_promedio - bandas_std, bandas_promedio + bandas_std,
                   color='blue', alpha=0.3)
ax[1].fill_between(wavelengths_nm, bandas_promedio_vegetacion - bandas_std_vegetacion, bandas_promedio_vegetacion + bandas_std_vegetacion,
                   color='green', alpha=0.3)
ax[1].fill_between(wavelengths_nm, bandas_promedio_suelo - bandas_std_suelo, bandas_promedio_suelo + bandas_std_suelo,
                   color='yellow', alpha=0.3)
ax[1].fill_between(wavelengths_nm, bandas_promedio_construido - bandas_std_construido, bandas_promedio_construido + bandas_std_construido,
                   color='violet', alpha=0.3)

# Dibujar líneas verticales para las bandas seleccionadas (según bandas_rgb)
colors = ['red', 'green', 'blue']
for i, band in enumerate(bandas_rgb):
    ax[1].axvline(x=wavelengths_nm[band], color=colors[i], alpha=0.3, linewidth=8)

# Configuración del gráfico espectral
ax[1].set_xlabel("Longitud de onda (nm)")
ax[1].set_ylabel("Valor promedio de la banda")
ax[1].set_title("Patrón espectral con bandas seleccionadas")
ax[1].legend()
ax[1].grid(True)

# Mostrar el gráfico
plt.tight_layout()
plt.show()
```